# β-VAE Training

## Clone Repository & Install Dependencies

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/nanokwok/Deep-Loan-Default-VAE.git'
REPO_DIR = '/content/Deep-Loan-Default-VAE'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('Repo already cloned — pulling latest')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU')


## Import data


In [ ]:
import os, json

# TODO: Change in colab and dont forgot to remove before push
KAGGLE_USERNAME = 'your_kaggle_username'
KAGGLE_KEY      = 'your_kaggle_api_key'

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(f'{kaggle_dir}/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
print('Kaggle credentials saved.')


In [ ]:
import os, subprocess, sys

RAW_DIR = 'data/raw'
RAW_CSV = f'{RAW_DIR}/accepted_2007_to_2018Q4.csv'
os.makedirs(RAW_DIR, exist_ok=True)

if os.path.exists(RAW_CSV):
    size_gb = os.path.getsize(RAW_CSV) / 1e9
    print(f'Raw CSV already present ({size_gb:.2f} GB) — skipping download.')
else:
    print('Downloading Lending Club dataset from Kaggle (~1.6 GB)...')
    # Dataset: https://www.kaggle.com/datasets/wordsforthewise/lending-club
    subprocess.run([
        sys.executable, '-m', 'kaggle', 'datasets', 'download',
        'wordsforthewise/lending-club',
        '--file', 'accepted_2007_to_2018Q4.csv',
        '-p', RAW_DIR, '--unzip',
    ], check=True)
    size_gb = os.path.getsize(RAW_CSV) / 1e9
    print(f'Download complete: {size_gb:.2f} GB')


## Run Preprocessing Pipeline

Runs `src/preprocess.py` to generate the `.npy` feature arrays.
Steps: load CSV → parse → ordinal encode → 70/15/15 split → 
impute → log1p → winsorize → OHE → VarianceThreshold → StandardScaler → save.


In [ ]:
import os, subprocess, sys, numpy as np, json
from pathlib import Path

PROC_DIR = Path('data/processed')
REQUIRED = [
    'train_features.npy', 'val_features.npy', 'val_labels.npy',
    'test_features.npy',  'test_labels.npy',  'scaler.pkl',
]

if all((PROC_DIR / f).exists() for f in REQUIRED):
    print('Processed arrays already exist — skipping. Delete data/processed/ to rerun.')
else:
    print('Running preprocessing (~8-12 min)...')
    subprocess.run([sys.executable, '-m', 'src.preprocess'], check=True)

train_arr = np.load(PROC_DIR / 'train_features.npy')
val_arr   = np.load(PROC_DIR / 'val_features.npy')
val_lbl   = np.load(PROC_DIR / 'val_labels.npy')
with open(PROC_DIR / 'feature_columns.json') as f:
    feat_names = json.load(f)

print(f'train : {train_arr.shape}  dtype={train_arr.dtype}')
print(f'val   : {val_arr.shape}  anomaly_rate={val_lbl.mean()*100:.1f}%')
print(f'input_dim = {train_arr.shape[1]}')
print(f'max|x|    = {abs(train_arr).max():.2f}  (target < 12 after log1p+winsorize)')
assert not np.isnan(train_arr).any(), 'NaN detected!'
print('NaN check passed.')


## Verify GPU & Data

In [ ]:
import torch, numpy as np
from pathlib import Path

device = (
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else 'cpu'
)
print(f'Device   : {device}')
if device == 'cuda':
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
elif device == 'cpu':
    print('WARNING: CPU only — training will be ~10× slower.')

proc = Path('data/processed')
train_arr = np.load(proc / 'train_features.npy')
val_lbl   = np.load(proc / 'val_labels.npy')
print(f'input_dim: {train_arr.shape[1]}')
print(f'train rows: {len(train_arr):,}')
print(f'val normal: {(val_lbl==0).sum():,}  anomaly: {(val_lbl==1).sum():,}')

# Preview model architecture before training
import sys; sys.path.insert(0, '.')
from src.model import BetaVAE
from src.config import BETA, LATENT_DIM
model = BetaVAE(input_dim=train_arr.shape[1])
n_params = sum(p.numel() for p in model.parameters())
print(f'BetaVAE  params={n_params:,}  latent_dim={LATENT_DIM}  beta={BETA}')


## Train the VAE

- Trains on normal (Fully Paid) rows only.
- Early stopping monitors **val normal-only** reconstruction loss (patience=10).
- Best checkpoint saved to `models/best_vae.pth`.
- Per-epoch log written to `reports/training_log.csv`.


In [ ]:
import sys, importlib
if '.' not in sys.path:
    sys.path.insert(0, '.')

import src.config as cfg
cfg.EXPERIMENTS_DIR = '/content/drive/MyDrive/Loan_VAE_Project'

for mod in list(sys.modules.keys()):
    if mod.startswith('src.'):
        del sys.modules[mod]

from src.train import train
exp_dir = train()   # returns Path to this run's experiment folder

print()
print('Experiment artefacts:', exp_dir)
import os
for f in sorted(os.listdir(str(exp_dir))):
    size = os.path.getsize(os.path.join(str(exp_dir), f))
    print(f'  {f:<30} {size/1024:.1f} KB')


In [ ]:
# Display experiment plots inline
from IPython.display import Image, display
import os

for fname in ['loss_curves.png', 'reconstruction_plot.png']:
    img_path = os.path.join(str(exp_dir), fname)
    if os.path.exists(img_path):
        print(f'--- {fname} ---')
        display(Image(img_path))
    else:
        print(f'{fname} not found.')


## Loss Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_df = pd.read_csv('reports/training_log.csv')
print(f'Epochs trained: {len(log_df)}')
print(log_df[['epoch','train_total','val_total','train_recon','val_recon','train_kl','val_kl']].tail(5).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
best_ep = log_df['val_total'].idxmin() + 1

for ax, tr_col, vl_col, title in zip(axes,
    ['train_total', 'train_recon', 'train_kl'],
    ['val_total',   'val_recon',   'val_kl'],
    ['Total Loss (recon + β·KL)', 'Reconstruction Loss (MSE)', 'KL Divergence'],
):
    ax.plot(log_df['epoch'], log_df[tr_col], label='train', color='steelblue')
    ax.plot(log_df['epoch'], log_df[vl_col], label='val (normal only)', color='crimson', linestyle='--')
    ax.axvline(best_ep, color='orange', linewidth=1.2, linestyle=':', label=f'best epoch {best_ep}')
    ax.set_xlabel('Epoch')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('β-VAE Training Curves', fontsize=13)
plt.tight_layout()
plt.savefig('reports/figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best epoch: {best_ep}  val_loss={log_df["val_total"].min():.4f}')


## Anomaly Score Preview

Reconstruction error distributions for Normal vs Anomaly rows in the val set.

**Expected:** Charged Off histogram shifted right of Fully Paid.
**If heavily overlapping:** lower `BETA` in `src/config.py` and retrain.


In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from src.model import BetaVAE
from src.config import DATA_PROC_DIR

proc   = Path(DATA_PROC_DIR)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt  = torch.load('models/best_vae.pth', map_location=device)
model = BetaVAE(input_dim=ckpt['input_dim']).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Checkpoint: epoch={ckpt['epoch']}  val_loss={ckpt['val_loss']:.4f}  β={ckpt['beta']}")

val_t   = torch.from_numpy(np.load(proc/'val_features.npy').astype('float32')).to(device)
val_lbl = np.load(proc / 'val_labels.npy')

with torch.no_grad():
    x_hat, _, _ = model(val_t)
    errors = ((val_t - x_hat) ** 2).mean(dim=1).cpu().numpy()

normal_e  = errors[val_lbl == 0]
anomaly_e = errors[val_lbl == 1]
print(f'Normal  mean={normal_e.mean():.4f}  p95={np.percentile(normal_e, 95):.4f}')
print(f'Anomaly mean={anomaly_e.mean():.4f}  p95={np.percentile(anomaly_e, 95):.4f}')
sep = anomaly_e.mean() / normal_e.mean()
print(f'Separation ratio (anomaly/normal mean): {sep:.2f}×  (>1.5 is good)')

clip = np.percentile(errors, 99)
bins = np.linspace(0, clip, 80)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(normal_e.clip(max=clip),  bins=bins, alpha=0.6, density=True, color='steelblue', label='Fully Paid (normal)')
ax.hist(anomaly_e.clip(max=clip), bins=bins, alpha=0.6, density=True, color='crimson',   label='Charged Off (anomaly)')
ax.set_xlabel('Reconstruction Error (mean squared per feature)')
ax.set_ylabel('Density')
ax.set_title('Anomaly Score Distribution — Val Set')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('reports/figures/anomaly_score_preview.png', dpi=150, bbox_inches='tight')
plt.show()
